In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.layers import Embedding, LSTM,GRU, Dense, Input,Bidirectional,SpatialDropout1D,Dropout

In [ ]:
file_path = '/kaggle/input/datasets/therohk/million-headlines/abcnews-date-text.csv'
df = pd.read_csv(file_path)

In [ ]:
text_data = df['headline_text'].head(100000).tolist()

In [ ]:
max_vocab = 8000
tokenizer = Tokenizer(num_words=max_vocab, oov_token="<OOV>")
tokenizer.fit_on_texts(text_data)

In [ ]:
print("Dataset mein total unique words the:", len(tokenizer.word_index) + 1)

In [ ]:
total_words = max_vocab + 1 
print("Total_vocab:", total_words)

In [ ]:
input_sequences = []
for sentence in text_data:
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]
    
    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i+1])

In [ ]:
max_len = max([len(x) for x in input_sequences])
print("Max length of sequence:", max_len)

padded_input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = padded_input_sequences[:, :-1]
y = padded_input_sequences[:, -1]

In [ ]:
callback=EarlyStopping(monitor='val_loss',patience=15,min_delta=0.001,baseline=None,restore_best_weights=True,mode="auto", verbose=1)

In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.00001
)

In [ ]:

embed_dim = 256  
input_len = max_len - 1

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(input_len,)),
    tf.keras.layers.Embedding(input_dim=total_words, output_dim=embed_dim),
    tf.keras.layers.SpatialDropout1D(0.2), 
    
    
    tf.keras.layers.GRU(256, return_sequences=True),
    tf.keras.layers.Dropout(0.2),
    
   
    tf.keras.layers.GRU(256, return_sequences=False), 
    tf.keras.layers.Dropout(0.2),
    
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(total_words, activation="softmax")
])

In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X, y, 
    epochs=30, 
    validation_split=0.2,batch_size=128,callbacks=[callback,reduce_lr]
)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(model.history.history['accuracy'],color='red',label='train')
plt.plot(model.history.history['val_accuracy'],color='blue',label='validation')
plt.legend()
plt.show()

In [ ]:
plt.plot(model.history.history['loss'],color='red',label='train')
plt.plot(model.history.history['val_loss'],color='blue',label='validation')
plt.legend()
plt.show()

In [ ]:
import time
import numpy as np

def sample_with_temperature(preds, temperature=0.6):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-7) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

text = "waht are"
for i in range(15):
    token_text = tokenizer.texts_to_sequences([text])[0]
    padded_token_text = pad_sequences([token_text], maxlen=11, padding='pre')
    
    preds = model.predict(padded_token_text, verbose=0)[0]
    
   
    next_index = sample_with_temperature(preds, temperature=0.6)
    predicted_word = tokenizer.index_word.get(next_index, "")
    
    if predicted_word and predicted_word != "<OOV>":
        text = text + " " + predicted_word
        print(text)
    else:
        break
    time.sleep(0.3)